In [9]:
# ==========================================
#  ENVIRONMENT SETUP & LIBRARY IMPORTS
# ==========================================
print("=================================================================")
print(" ENVIRONMENT SETUP & LIBRARY IMPORTS")
print("=================================================================")

import spacy
import numpy as np
import pandas as pd
from numpy.linalg import norm

# Load the medium-sized English model which includes multi-dimensional word vectors
# (Small models like 'en_core_web_sm' do not include vectors)
nlp = spacy.load("en_core_web_md")

print("Status: Libraries imported and spaCy medium model loaded successfully.")

 ENVIRONMENT SETUP & LIBRARY IMPORTS
Status: Libraries imported and spaCy medium model loaded successfully.


In [10]:
# ==========================================
#  EXERCISES 2, 3, & 4: TOKEN VECTOR PROPERTIES
# ==========================================
print("=================================================================")
print("EXERCISES 2, 3, & 4: TOKEN VECTOR PROPERTIES")
print("=================================================================")

# Define the target sentence
doc1 = nlp("Bahi takes a lunch at STGI.")

print(f"{'TOKEN':<10} | {'HAS VECTOR?':<12} | {'IS OOV?':<10} | {'VECTOR SHAPE'}")
print("-" * 55)

# Inspect whether tokens have vectors, if they are Out-Of-Vocabulary, and their shape
for token in doc1:
    print(f"{token.text:<10} | {str(token.has_vector):<12} | {str(token.is_oov):<10} | {token.vector.shape}")

EXERCISES 2, 3, & 4: TOKEN VECTOR PROPERTIES
TOKEN      | HAS VECTOR?  | IS OOV?    | VECTOR SHAPE
-------------------------------------------------------
Bahi       | False        | True       | (300,)
takes      | True         | False      | (300,)
a          | True         | False      | (300,)
lunch      | True         | False      | (300,)
at         | True         | False      | (300,)
STGI       | False        | True       | (300,)
.          | True         | False      | (300,)


In [5]:
# ==========================================
#  EXERCISES 5 & 6: SIMILARITY TO BASE TOKEN
# ==========================================
print("=================================================================")
print("EXERCISES 5 & 6: SIMILARITY TO BASE TOKEN")
print("=================================================================")

# Define the base token and check its shape
base_token = nlp("meal")[0]
print(f"Base Token: '{base_token.text}' | Vector Shape: {base_token.vector.shape}\n")

print("=== SIMILARITY TO 'meal' ===")

# Compute similarity between 'meal' and all tokens in the doc
for token in doc1:
    if not token.is_punct:
        # Check if vector exists to prevent the [W008] warning on OOV words like Bahi and STGI
        if token.has_vector:
            similarity_score = token.similarity(base_token)
        else:
            similarity_score = 0.0000  # Default to 0 for out-of-vocabulary words

        print(f"'{token.text:<10}' similarity: {similarity_score:.4f}")

EXERCISES 5 & 6: SIMILARITY TO BASE TOKEN
Base Token: 'meal' | Vector Shape: (300,)

=== SIMILARITY TO 'meal' ===
'Bahi      ' similarity: 0.0000
'takes     ' similarity: 0.3308
'a         ' similarity: 0.2647
'lunch     ' similarity: 0.4670
'at        ' similarity: 0.0212
'STGI      ' similarity: 0.0000


In [12]:
# ==========================================
#  EXERCISES 7 & 8: CROSS-SENTENCE WORD SIMILARITY MATRIX
# ==========================================
print("=================================================================")
print("EXERCISES 7 & 8: CROSS-SENTENCE WORD SIMILARITY MATRIX")
print("=================================================================")

def compare_sentences(sent1, sent2):
    """
    Computes a cross-sentence word similarity matrix while safely
    handling out-of-vocabulary words to prevent [W008] warnings.
    """
    d1 = nlp(sent1)
    d2 = nlp(sent2)

    similarity_matrix = []
    for t1 in d1:
        row = []
        for t2 in d2:
            # Check if both tokens have valid vectors before calculating similarity
            if t1.has_vector and t2.has_vector:
                row.append(t1.similarity(t2))
            else:
                row.append(0.000)  # Default to 0.0 if either word is OOV
        similarity_matrix.append(row)

    df = pd.DataFrame(
        similarity_matrix,
        index=[t.text for t in d1],
        columns=[t.text for t in d2]
    )
    return df

# Define the sentences and output the resulting matrix
s1 = "Bahi takes a lunch at STGI"
s2 = "Bahi takes a lunch at home"

similarity_df = compare_sentences(s1, s2)
print(similarity_df.round(3))

EXERCISES 7 & 8: CROSS-SENTENCE WORD SIMILARITY MATRIX
       Bahi  takes      a  lunch     at   home
Bahi    0.0  0.000  0.000  0.000  0.000  0.000
takes   0.0  1.000  0.373  0.213  0.135  0.156
a       0.0  0.373  1.000  0.267  0.017  0.194
lunch   0.0  0.213  0.267  1.000  0.098  0.045
at      0.0  0.135  0.017  0.098  1.000  0.057
STGI    0.0  0.000  0.000  0.000  0.000  0.000


In [13]:
# ==========================================
# CELL 5: EXERCISE 9: SPECIFIC WORD CATEGORY SIMILARITY
# ==========================================
print("=================================================================")
print("EXERCISE 9: SPECIFIC WORD CATEGORY SIMILARITY")
print("=================================================================")

target_word = nlp("iphone")[0]
comparison_doc = nlp("apple samsung iphone dog kitten")

print(f"SIMILARITY COMPARISON: '{target_word.text}'")
print("-" * 45)

for token in comparison_doc:
    score = target_word.similarity(token)
    print(f"iphone vs {token.text:<10} : {score:.4f}")

EXERCISE 9: SPECIFIC WORD CATEGORY SIMILARITY
SIMILARITY COMPARISON: 'iphone'
---------------------------------------------
iphone vs apple      : 0.1989
iphone vs samsung    : 0.3120
iphone vs iphone     : 1.0000
iphone vs dog        : 0.1561
iphone vs kitten     : 0.1561


In [14]:
# ==========================================
# CELL 6: EXERCISES 10, 11, & 12: VECTOR MATHEMATICS
# ==========================================
print("=================================================================")
print("EXERCISES 10, 11, & 12: VECTOR MATHEMATICS")
print("=================================================================")

# 10: Extract raw vector arrays for our target words
king = nlp("king")[0].vector
man = nlp("man")[0].vector
woman = nlp("woman")[0].vector
queen = nlp("queen")[0].vector

print("=== 10: VECTOR SHAPES ===")
print(f"King:  {king.shape}")
print(f"Man:   {man.shape}")
print(f"Woman: {woman.shape}")
print(f"Queen: {queen.shape}\n")

# 11: Compute vector arithmetic: King - Man + Woman
result_vector = king - man + woman
print("=== 11: VECTOR MATH RESULT ===")
print(f"Shape of (King - Man + Woman): {result_vector.shape}\n")

# Helper function to compute cosine similarity for raw numpy arrays
def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (norm(v1) * norm(v2))

# 12: Compute cosine similarity between the mathematical result and "queen"
final_similarity = cosine_similarity(result_vector, queen)

print("=== 12: FINAL COSINE SIMILARITY ===")
print(f"Similarity between (King - Man + Woman) and (Queen): {final_similarity:.4f}")

EXERCISES 10, 11, & 12: VECTOR MATHEMATICS
=== 10: VECTOR SHAPES ===
King:  (300,)
Man:   (300,)
Woman: (300,)
Queen: (300,)

=== 11: VECTOR MATH RESULT ===
Shape of (King - Man + Woman): (300,)

=== 12: FINAL COSINE SIMILARITY ===
Similarity between (King - Man + Woman) and (Queen): 0.4812
